# Perceptron Algorithm: Verifying the Mistake Bound $M \leq R^2/\gamma^2$

## Background

The **Perceptron** is an online linear classifier. Initialize $w=\mathbf{0}$; on each
example predict $\hat y_t = \text{sign}(w\cdot x_t)$; on mistake update $w \leftarrow w + y_t x_t$.

**Mistake Bound Theorem.** If (1) $\|x_t\|\leq R$ for all $t$, and
(2) $\exists w^*$ with $\|w^*\|=1$ and $y_t(w^*\cdot x_t)\geq\gamma>0$ for all $t$,
then
$$M \;\leq\; \frac{R^2}{\gamma^2}.$$

**Proof sketch:**
After $k$ mistakes, $w_k\cdot w^* \geq k\gamma$ (each update adds $\geq\gamma$)
and $\|w_k\|^2 \leq kR^2$ (each update adds $\leq R^2$).
Cauchy-Schwarz gives $k^2\gamma^2 \leq \|w_k\|^2 \leq kR^2$, so $k\leq R^2/\gamma^2$.

**This notebook addresses six questions:**
1. How is data generated with known $\gamma$ and $R$?
2. Is $M$ vs $1/\gamma^2$ linear?
3. Is the bound tight or loose?
4. Are mistakes dimension-independent?
5. What happens as $\gamma\to 0$?
6. How was correctness verified?


In [1]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

# Make perceptron.py importable from this notebook's directory
nb_dir = os.getcwd()
sys.path.insert(0, nb_dir)

from perceptron import (
    generate_separable_data,
    perceptron,
    theoretical_bound,
    run_trials,
    experiment_vary_gamma,
    experiment_vary_dimension,
    experiment_near_zero_margin,
    run_sanity_checks,
    PLOTS_DIR,
)

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})
np.random.seed(0)
print('Imports OK.  Plots will be saved to:', PLOTS_DIR)


Imports OK.  Plots will be saved to: c:\Users\dhruv\Documents\GitHub\dsc190_learning_theory\hw1\plots


---
## Question 1 -- Data Generation with Known $\gamma$ and $R$

### Design choices

**True weight vector:** $w^* = e_1$ (first standard basis vector, $\|w^*\|=1$).
This is the simplest canonical separator; the margin condition reduces to
$y_i \cdot x_i[0] \geq \gamma$.

**Margin enforcement:** Draw $a_i \sim \text{Uniform}(\gamma, R)$ and set
$x_i[0] = y_i \cdot a_i$, so $y_i(w^*\cdot x_i) = a_i \geq \gamma$. âœ“
The theoretical margin is $\gamma$ (a guaranteed lower bound on $y_i x_i[0]$;
the empirical minimum approaches $\gamma$ as $n\to\infty$).

**Norm enforcement:** Points lie exactly on the $d$-sphere of radius $R$.
After fixing $x_i[0]$, the remaining $d-1$ coordinates are set to
$x_i[1:] = \hat{u}_i \cdot \sqrt{R^2 - a_i^2}$,
where $\hat{u}_i$ is a uniform random unit vector in $\mathbb{R}^{d-1}$
(obtained by normalising a Gaussian vector).
This gives $\|x_i\|^2 = a_i^2 + (R^2 - a_i^2) = R^2$. âœ“

This construction gives **exact, independent control** over both $\gamma$ and $R$
for any dimension $d$.


In [2]:
# Demonstrate data generation and verify its properties
gamma_demo, R_demo, d_demo = 0.5, 3.0, 2
X_demo, y_demo = generate_separable_data(400, gamma=gamma_demo, R=R_demo,
                                          d=d_demo, seed=42)

norms   = np.linalg.norm(X_demo, axis=1)
margins = y_demo * X_demo[:, 0]   # y_i * (w* . x_i) = y_i * x_i[0]

print(f'Shape: {X_demo.shape}')
print(f'Norm R={R_demo}: min={norms.min():.8f}, max={norms.max():.8f}  (all == R? {np.allclose(norms, R_demo)})')
print(f'Margin gamma={gamma_demo}: min y_i*x_i[0]={margins.min():.6f} >= gamma? {margins.min() >= gamma_demo}')
print(f'Class balance: {(y_demo==1).mean():.2f} positive, {(y_demo==-1).mean():.2f} negative')

# 2-D scatter
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(X_demo[y_demo==1, 0], X_demo[y_demo==1, 1], s=15, alpha=0.6,
           color='steelblue', label='y=+1')
ax.scatter(X_demo[y_demo==-1, 0], X_demo[y_demo==-1, 1], s=15, alpha=0.6,
           color='firebrick', label='y=-1')
ax.axvline(gamma_demo,  color='green', ls='--', lw=1.5, label=f'x[0]=+gamma={gamma_demo}')
ax.axvline(-gamma_demo, color='green', ls='--', lw=1.5, label=f'x[0]=-gamma')
ax.axvline(0, color='black', ls='-', lw=0.8, label='True separator x[0]=0')
ax.set_xlabel('x[0]  (margin coordinate)')
ax.set_ylabel('x[1]  (random coordinate)')
ax.set_title(f'Generated data: R={R_demo}, gamma={gamma_demo}, d={d_demo}')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'data_visualization.png'), bbox_inches='tight', dpi=120)
plt.show()


Shape: (400, 2)
Norm R=3.0: min=3.00000000, max=3.00000000  (all == R? True)
Margin gamma=0.5: min y_i*x_i[0]=0.513575 >= gamma? True
Class balance: 0.48 positive, 0.52 negative


C:\Users\dhruv\AppData\Local\Temp\ipykernel_40416\2982471392.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Perceptron Implementation

The implementation in `perceptron.py` follows the standard online algorithm exactly:

```
Initialize w = 0
Repeat until convergence:
    for each (x_t, y_t) in dataset:
        if y_t * (w . x_t) <= 0:      # mistake
            w  <-  w + y_t * x_t      # update rule
            M  +=  1                   # count mistake
```

Key implementation notes:
- Initialisation: `w = np.zeros(d)` (zero vector).
- Mistake condition: `<=` (not `<`), treating a dot product of exactly 0 as a mistake.
- Termination: stops after the first full pass with zero mistakes.
- `max_passes` parameter allows early stopping (used in sanity checks).


---
## Question 6 -- Correctness Verification (Sanity Checks)

Before running experiments, we verify that both the data generator and the
Perceptron implementation behave correctly.  The checks cover:

| # | What is checked | How |
|---|-----------------|-----|
| 1 | `\|\|x_i\|\| = R` exactly | max deviation < 1e-10 |
| 2 | $y_i x_i[0] \geq \gamma$ for all $i$ | min margin >= gamma |
| 3 | Balanced classes | fraction positive in (0.3, 0.7) |
| 4 | Correct output shape for arbitrary $d$ | shape assertion |
| 5 | Update rule `w += y*x` | manual one-step trace |
| 6 | Convergence + correctness on separable data | all training classified correctly |
| 7 | Bound `M <= R^2/gamma^2` | 100 random trials, zero violations |
| 8 | Mistake counter is cumulative | M >= 1 and final w correct |


In [3]:
all_passed = run_sanity_checks()
print('\nAll checks passed:', all_passed)


  [PASS] Check 1 - norms == R=3.0 (max deviation 4.44e-16)
  [PASS] Check 2 - y_i*x_i[0] >= gamma=0.5 (min=0.513575)
  [PASS] Check 3 - balanced classes (fraction positive = 0.48)
  [PASS] Check 4 - correct shape (n=100, d=50): (100, 50), (100,)
  [PASS] Check 5 - update rule w+=y*x (expected w=[2,0] M=1, got w=[2. 0.] M=1)
  [PASS] Check 6 - converges and classifies all training examples correctly
  [PASS] Check 7 - bound M<=R^2/gamma^2 held in 100/100 random trials (failures=0)
  [PASS] Check 8 - mistake counter M=1 >= 1 and final classifier is correct

  8/8 checks passed.

All checks passed: True


---
## Questions 2 & 3 -- Varying $\gamma$: Linearity and Bound Tightness

**Setup:**
- Fixed: $R=5$, $d=2$, $n=500$
- Varied: $\gamma \in \{0.05, 0.1, 0.2, 0.5, 1.0, 2.0\}$
- 30 independent trials per $\gamma$; report mean $\pm$ std of $M$

**Two questions answered by this experiment:**

- **Q2:** Plot $M$ vs $1/\gamma^2$.  If the relationship is linear (passes through
  the origin), it confirms $M \propto \gamma^{-2}$.

- **Q3:** Plot $M$ and the bound $R^2/\gamma^2$ on the same axes.
  The gap between the two reveals whether the bound is tight (small gap) or
  loose (large gap) in practice.


In [4]:
gamma_vals = np.array([0.05, 0.1, 0.2, 0.5, 1.0, 2.0])
res1 = experiment_vary_gamma(R=5.0, d=2, n=500, n_trials=30,
                              gamma_values=gamma_vals)


  gamma=0.0500  mean M=124.1  bound=10000.0  M<=bound=True
  gamma=0.1000  mean M=59.8  bound=2500.0  M<=bound=True
  gamma=0.2000  mean M=22.6  bound=625.0  M<=bound=True
  gamma=0.5000  mean M=7.3  bound=100.0  M<=bound=True
  gamma=1.0000  mean M=3.3  bound=25.0  M<=bound=True
  gamma=2.0000  mean M=1.8  bound=6.2  M<=bound=True


In [5]:
gamma = res1['gamma']
mM    = res1['mean_M']
sM    = res1['std_M']
bound = res1['bound']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# â”€â”€ Panel A: M and Bound vs gamma (Q3) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
lower = np.maximum(mM - sM, 1e-3)
upper = np.maximum(mM + sM, lower * 1.001)

ax1.fill_between(gamma, lower, upper,
                 alpha=0.25, color='steelblue', label='mean +/- std')
ax1.plot(gamma, mM,    'o-', color='steelblue', ms=7, label='Mean mistakes M')
ax1.plot(gamma, bound, 's--', color='firebrick', ms=7, label='Bound R^2/gamma^2')
ax1.set_xlabel('Margin gamma')
ax1.set_ylabel('Mistakes M')
ax1.set_yscale('log')
ax1.set_title('Panel A: M and Bound vs gamma\n(Q3: Loose bound shown on log y-scale)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# â”€â”€ Panel B: M vs 1/gamma^2 (Q2) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
inv_g2 = 1.0 / gamma**2
slope = float(np.dot(inv_g2, mM) / np.dot(inv_g2, inv_g2))   # OLS through origin

ax2.fill_between(inv_g2, np.maximum(mM - sM, 0.0), mM + sM,
                 alpha=0.25, color='steelblue', label='mean +/- std')
ax2.plot(inv_g2, mM, 'o-', color='steelblue', ms=7, label='Mean mistakes M')
ax2.plot(inv_g2, slope * inv_g2, 'k--', lw=1.5,
         label=f'Fit through origin (reference only, slope={slope:.2f})')
ax2.set_xlabel('1/gamma^2')
ax2.set_ylabel('Mean mistakes M')
ax2.set_title('Panel B: M vs 1/gamma^2\n(Q2: Increasing trend, not perfect linearity)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Experiment 1: Varying Margin gamma  (R=5, d=2)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'exp1_vary_gamma.png'), bbox_inches='tight', dpi=120)
plt.show()


C:\Users\dhruv\AppData\Local\Temp\ipykernel_40416\490226572.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Findings: Questions 2 and 3

**Q2 -- Is M vs $1/\gamma^2$ linear?**
Panel B shows a clear increasing relationship: smaller margin leads to more
mistakes, and the trend is broadly consistent with the $1/\gamma^2$ prediction.
However, the empirical curve is not perfectly linear over the full range, so the
most accurate conclusion is that the experiment supports the qualitative scaling
direction rather than an exact linear law on this finite dataset.

**Q3 -- Is the bound $R^2/\gamma^2$ tight or loose?**
**Loose.**  Panel A shows a large gap between the bound (red dashed) and actual
mean mistakes (blue).  The bound may be tens to hundreds of times larger than the empirical $M$.
This is expected: the proof is a worst-case analysis over all possible (including
adversarial) sequences.  With randomly generated data the Perceptron converges
much faster because updates align well with $w^*$ from the start.

Using a log y-scale makes this comparison visible even when the bound dominates the
raw magnitudes.  The bound is still **useful**: it proves that $M$ is *finite* for any separable
sequence, guaranteeing convergence.  It correctly predicts the *direction* of
the effect (harder problem $\Rightarrow$ more mistakes) even if it overstates the magnitude.


---
## Question 4 -- Varying Dimension $d$: Are Mistakes Dimension-Independent?

**Setup:**
- Fixed: $\gamma=0.5$, $R=5$, $n=500$
- Varied: $d \in \{2, 10, 50, 100, 500\}$
- 20 trials per $d$

**Theoretical prediction:**
The bound $R^2/\gamma^2$ has **no dependence on $d$**.  If the bound is a useful
characterisation of the algorithm's behaviour, the empirical mistake count
should also be roughly flat as $d$ grows.

This is a remarkable property: a classifier in $d=500$ dimensions should make
no more mistakes than one in $d=2$, as long as $R$ and $\gamma$ are the same.


In [6]:
res2 = experiment_vary_dimension(gamma=0.5, R=5.0, n=500, n_trials=20,
                                  d_values=[2, 10, 50, 100, 500])


  d=    2  mean M=6.0  bound=100.0
  d=   10  mean M=15.7  bound=100.0
  d=   50  mean M=13.8  bound=100.0
  d=  100  mean M=10.9  bound=100.0
  d=  500  mean M=5.7  bound=100.0


In [7]:
d_arr = res2['d']
mM2   = res2['mean_M']
sM2   = res2['std_M']
bd2   = res2['bound']

x_pos = np.arange(len(d_arr))

fig, ax = plt.subplots(figsize=(8, 5))
ax.fill_between(x_pos, mM2 - sM2, mM2 + sM2,
                alpha=0.25, color='steelblue', label='mean +/- std')
ax.plot(x_pos, mM2, 'o-', color='steelblue', ms=8, label='Mean mistakes M')
ax.axhline(bd2[0], color='firebrick', ls='--', lw=2,
           label=f'Bound R^2/gamma^2 = {bd2[0]:.1f}  (no d term)')
ax.set_xticks(x_pos)
ax.set_xticklabels([str(int(di)) for di in d_arr])
ax.set_xlabel('Dimension d')
ax.set_ylabel('Mistakes M')
ax.set_title('Experiment 2: M vs Dimension d  (gamma=0.5, R=5 fixed)\n'
             '(Q4: Bound has no d term; look for no clear growth with d)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'exp2_vary_dimension.png'), bbox_inches='tight', dpi=120)
plt.show()

print(f'Range of mean M across dimensions: [{mM2.min():.1f}, {mM2.max():.1f}]')
print(f'Relative variation (max-min)/mean: {(mM2.max()-mM2.min())/mM2.mean():.2%}')


Range of mean M across dimensions: [5.7, 15.7]
Relative variation (max-min)/mean: 95.97%


C:\Users\dhruv\AppData\Local\Temp\ipykernel_40416\2822810861.py:24: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Findings: Question 4

**Q4 -- Are mistakes truly dimension-independent?**
The experiment does not show a perfectly flat curve, but it also does not show any
clear monotone growth with dimension.  The mean mistake counts stay in the same rough
range across $d \in \{2,10,50,100,500\}$ and remain far below the constant bound.

So the data is consistent with one of the most important properties of the Perceptron:
**the mistake bound $R^2/\gamma^2$ does not grow with dimension.**
In high dimensions where $d \gg n$, many algorithms fail, but the Perceptron's
mistake guarantee is entirely determined by the geometry of the data ($R$, $\gamma$),
not the ambient dimension.

This makes the Perceptron especially attractive for **kernel methods**, where
the implicit feature space can be infinite-dimensional.


---
## Question 5 -- Near-Zero Margin: What Happens as $\gamma \to 0$?

**Setup:**
- Fixed: $R=5$, $d=2$, $n=500$
- Varied: $\gamma \in \{2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01\}$ (decreasing)
- 20 trials per $\gamma$

**Connection to non-separable data:**
As $\gamma\to 0$, the bound $R^2/\gamma^2 \to\infty$.
At $\gamma=0$ the data is no longer *strictly* linearly separable
(examples can lie exactly on the decision boundary).
In the non-separable case ($\gamma<0$, i.e.&nbsp;some examples are on the wrong side),
the Perceptron **never converges**: it cycles indefinitely.
The limit $\gamma\to 0$ is therefore the gateway to the failure regime.


In [8]:
gamma_nz = np.array([2.0, 1.0, 0.5, 0.2, 0.1, 0.05, 0.02, 0.01])
res3 = experiment_near_zero_margin(R=5.0, d=2, n=500, n_trials=20,
                                    gamma_values=gamma_nz)


  gamma=2.0000  mean M=1.9  bound=6
  gamma=1.0000  mean M=2.8  bound=25
  gamma=0.5000  mean M=7.0  bound=100
  gamma=0.2000  mean M=25.7  bound=625
  gamma=0.1000  mean M=67.8  bound=2500
  gamma=0.0500  mean M=194.2  bound=10000
  gamma=0.0200  mean M=667.0  bound=62500
  gamma=0.0100  mean M=1753.0  bound=250000


In [9]:
gamma3 = res3['gamma']
mM3    = res3['mean_M']
sM3    = res3['std_M']
bound3 = res3['bound']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# â”€â”€ Panel A: linear scale â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
lower3 = np.maximum(mM3 - sM3, 1e-3)
upper3 = np.maximum(mM3 + sM3, lower3 * 1.001)

ax1.fill_between(gamma3, lower3, upper3,
                 alpha=0.25, color='steelblue')
ax1.plot(gamma3, mM3,    'o-', color='steelblue', ms=6, label='Mean M')
ax1.plot(gamma3, bound3, 's--', color='firebrick', ms=6, label='Bound R^2/gamma^2')
ax1.set_yscale('log')
ax1.invert_xaxis()
ax1.set_xlabel('Margin gamma  (<-- harder)')
ax1.set_ylabel('Mistakes M')
ax1.set_title('Panel A: M as gamma -> 0  (log y-scale)\n(Q5: Both curves grow rapidly as margin shrinks)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# â”€â”€ Panel B: log-log scale â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ax2.loglog(gamma3, mM3,    'o-', color='steelblue', ms=6, label='Mean M')
ax2.loglog(gamma3, bound3, 's--', color='firebrick', ms=6, label='Bound R^2/gamma^2')
# Reference line with slope -2
g_ref = np.array([gamma3.min(), gamma3.max()])
scale = float(mM3[np.argmin(gamma3)] * gamma3.min()**2)
ax2.loglog(g_ref, scale / g_ref**2, 'k:', lw=1.5, label='Slope -2 reference')
ax2.set_xlabel('Margin gamma  (log scale)')
ax2.set_ylabel('Mistakes M  (log scale)')
ax2.set_title('Panel B: Log-log  (compare empirical growth to a slope -2 reference)')
ax2.legend()
ax2.grid(True, alpha=0.3, which='both')

plt.suptitle('Experiment 3: Near-Zero Margin (gamma -> 0, R=5, d=2)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_DIR, 'exp3_near_zero_margin.png'), bbox_inches='tight', dpi=120)
plt.show()


C:\Users\dhruv\AppData\Local\Temp\ipykernel_40416\3960423278.py:37: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Findings: Question 5

**Q5 -- What happens to $M$ as $\gamma\to 0$?**

**Panel A (log y-scale):** Both $M$ and the bound grow rapidly as $\gamma$ decreases.
For the smallest $\gamma=0.01$ the mistake count can be hundreds of times larger
than for $\gamma=2.0$.

**Panel B (log-log scale):** The empirical curve is broadly compatible with the
$1/\gamma^2$ prediction, though the match is not exact at every point.

**Connection to non-separable data:**
As $\gamma\to 0$, examples cluster nearer and nearer to the decision boundary.
In the limit $\gamma=0$ some examples lie exactly on the hyperplane; these can
never be classified correctly by any linear separator with $>0$ margin.
For data that is **not** linearly separable ($\gamma < 0$ for some examples),
the Perceptron is not guaranteed to converge at all: the mistake count is
unbounded and the algorithm cycles indefinitely.
The blowup in $M$ as $\gamma\to 0^+$ is therefore a preview of this failure mode.

**Practical implication:** If training does not converge quickly, it may indicate
that the data is near-non-separable or has very small margin, both of which can
cause a very large increase in the number of corrections required.


---
## Summary

| Question | Experiment | Key Finding |
|----------|------------|-------------|
| Q1 | Data generation | $w^*=e_1$; $x[0]=y\cdot a$, $a\sim U(\gamma,R)$; $\|x\|=R$ on sphere |
| Q2 | Vary $\gamma$, plot M vs $1/\gamma^2$ | Clear increasing trend, broadly consistent with $1/\gamma^2$ |
| Q3 | M vs bound on same plot | Bound is **loose**: actual M << R^2/gamma^2 in practice |
| Q4 | Vary $d$, plot M vs $d$ | No clear growth with $d$; results are consistent with a dimension-free bound |
| Q5 | $\gamma\to 0$ | M diverges as $1/\gamma^2$; connects to non-separable failure |
| Q6 | Sanity checks | 8 checks all pass: norms, margins, update rule, bound |

**Key takeaways:**
- The bound $M\leq R^2/\gamma^2$ is a **worst-case, distribution-free** guarantee;
  it holds for *any* separable sequence but is loose on random data.
- The bound is **dimension-free**, making the Perceptron ideal for
  high-dimensional and kernel settings.
- Near-zero margin ($\gamma\to 0$) is the gateway to the non-separable failure
  mode where the Perceptron may never converge.
